# 5 · Fine-tuning Wav2Vec2 XLS-R-300M  (GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chemvatho/Koelsch-Phoneme-Recognition/blob/main/05_finetune/05_wav2vec2_finetune.ipynb)

**Pipeline stage 5 of 6.** Fine-tune `facebook/wav2vec2-xls-r-300m` with a CTC
head for Kölsch **phoneme recognition**.

> **Run this on a CUDA GPU.** On Colab: *Runtime → Change runtime type → GPU
> (T4 is enough)*. A full 150-epoch run is impractical on CPU / Apple-Silicon
> MPS (the CTC loss op isn't implemented on MPS).

## Setup

In [ ]:
!pip -q install "transformers>=4.40" datasets evaluate jiwer torchaudio accelerate librosa
import os
# Harmless on GPU; lets CTC fall back to CPU if you ever run on Apple-Silicon MPS.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import torch, json, numpy as np
from dataclasses import dataclass
from typing import Dict, List, Union

USE_CUDA = torch.cuda.is_available()
device = "cuda" if USE_CUDA else ("mps" if getattr(torch.backends,"mps",None)
          and torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "·", device)
if USE_CUDA:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NOTE: no CUDA GPU detected. On Colab: Runtime -> Change runtime type -> GPU.")
    print("      This notebook is tuned for GPU; on CPU/MPS it will be very slow.")

In [ ]:
# === Portable setup — identical paths in VS Code, Jupyter & Google Colab ===
import sys
from pathlib import Path
try:
    import google.colab  # noqa: F401
    _here = any((Path(p)/"kolsch_paths.py").exists() for p in [Path.cwd(), *Path.cwd().parents])
    if not _here and not Path("/content/kolsch-tandem/kolsch_paths.py").exists():
        os.system("git clone -q https://github.com/chemvatho/Koelsch-Phoneme-Recognition.git /content/kolsch-tandem")
except Exception:
    pass
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"kolsch_paths.py").exists()),
             Path("/content/kolsch-tandem"))
sys.path.insert(0, str(_root))
from kolsch_paths import ROOT, DATA, PAGES, AUDIO, TRANS, SEG, INDEX, LEXICON, MODELS
os.chdir(ROOT)
print("repo root:", ROOT)

## 1 · Load the manifest + speaker-disjoint split

Reads `data/segments/manifest.csv` (from **NB3**, phonemised by **NB4**). Each
row needs `audio_path` and `phonetic` (`p h | p h`).

In [ ]:
from datasets import Dataset
import pandas as pd, hashlib

MANIFEST = os.path.join(SEG, "manifest.csv")
assert os.path.exists(MANIFEST), (
    f"{MANIFEST} not found — run Notebook 3 (segmentation) then Notebook 4 "
    "(normalisation) first, so the manifest has audio_path + phonetic columns.")
man = pd.read_csv(MANIFEST)
assert "phonetic" in man.columns, "manifest has no 'phonetic' column — run Notebook 4."
man = man.merge(pd.read_csv(INDEX)[["id","speaker"]], on="id", how="left")

def bucket(speaker):
    h = int(hashlib.md5(str(speaker).encode()).hexdigest(), 16) % 10
    return "test" if h < 1 else "valid" if h < 2 else "train"

speakers = man["speaker"].fillna("unknown").unique()
if len(speakers) >= 3:
    man["split"] = man["speaker"].map(bucket)           # speaker-disjoint (~80/10/10)
else:
    rng = np.random.default_rng(0); r = rng.random(len(man))
    man["split"] = np.where(r < 0.8, "train", np.where(r < 0.9, "valid", "test"))
if len(man) > 1 and (man["split"] == "valid").sum() == 0:
    man.loc[man.index[-1], "split"] = "valid"

def to_ds(split):
    sub = man[man["split"] == split][["audio_path", "phonetic"]]
    return Dataset.from_pandas(sub, preserve_index=False)   # audio decoded in prepare()

train_ds, valid_ds, test_ds = to_ds("train"), to_ds("valid"), to_ds("test")
print("split:", {s: int((man["split"] == s).sum()) for s in ["train","valid","test"]})

## 2 · Phoneme vocabulary + processor

In [ ]:
from transformers import (Wav2Vec2PhonemeCTCTokenizer, Wav2Vec2FeatureExtractor,
                          Wav2Vec2Processor)

VOCAB_PATH = os.path.join(MODELS, "vocab.json")
def build_vocab(phonetic_series, path=VOCAB_PATH):
    toks = set()
    for s in phonetic_series: toks.update(str(s).split())
    toks.discard("|")
    vocab = {t: i for i, t in enumerate(sorted(toks))}
    vocab["|"] = len(vocab); vocab["[UNK]"] = len(vocab); vocab["[PAD]"] = len(vocab)
    json.dump(vocab, open(path, "w"), ensure_ascii=False)
    return vocab

vocab = build_vocab(man[man["split"] == "train"]["phonetic"])
tokenizer = Wav2Vec2PhonemeCTCTokenizer(VOCAB_PATH, unk_token="[UNK]",
                pad_token="[PAD]", word_delimiter_token="|",
                phone_delimiter_token=" ", do_phonemize=False)
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000,
                padding_value=0.0, do_normalize=True, return_attention_mask=True)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
print(f"vocab: {len(vocab)} tokens ->", VOCAB_PATH)

## 3 · Prepare dataset (load audio → input_values, phonetic → labels)

In [ ]:
import soundfile as sf
try:
    import librosa
except Exception:
    librosa = None

def _load_16k(path):
    """Load a wav as mono float32 at 16 kHz (no datasets Audio cast -> version-robust)."""
    wav, sr = sf.read(path)
    if getattr(wav, "ndim", 1) > 1: wav = wav.mean(axis=1)
    wav = np.asarray(wav, dtype=np.float32)
    if sr != 16000:
        if librosa is None: raise RuntimeError("pip install librosa to resample")
        wav = librosa.resample(wav, orig_sr=sr, target_sr=16000)
    return wav

def prepare(batch):
    wav = _load_16k(batch["audio_path"])
    batch["input_values"] = processor(wav, sampling_rate=16000).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    batch["labels"] = processor(text=batch["phonetic"]).input_ids
    return batch

train_ds = train_ds.map(prepare, remove_columns=train_ds.column_names)
valid_ds = valid_ds.map(prepare, remove_columns=valid_ds.column_names)
print("prepared:", len(train_ds), "train /", len(valid_ds), "valid")

## 4 · CTC data collator

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids":   f["labels"]}        for f in features]
        batch = self.processor.feature_extractor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")
        batch["labels"] = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)

## 5 · Metrics — WER & CER (phoneme-level)

In [ ]:
import evaluate
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    ids = np.argmax(logits, axis=-1)
    labels = pred.label_ids
    labels[labels == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(ids)
    label_str = processor.batch_decode(labels, group_tokens=False)
    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str),
            "cer": cer_metric.compute(predictions=pred_str, references=label_str)}

## 6 · Model + GPU training configuration

In [ ]:
from transformers import Wav2Vec2ForCTC, TrainingArguments, Trainer

model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    attention_dropout=0.05, hidden_dropout=0.05, feat_proj_dropout=0.05,
    mask_time_prob=0.05, layerdrop=0.05,
    ctc_loss_reduction="mean", ctc_zero_infinity=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer))
model.freeze_feature_encoder()
model.gradient_checkpointing_enable()

# Version-robust: transformers 5.0 removed group_by_length (and may remove
# more). Filter against the installed signature instead of pinning a version.
import inspect
_want = dict(
    output_dir=os.path.join(MODELS, "kolsch_wav2vec2_model"),
    group_by_length=True,
    per_device_train_batch_size=8,        # T4-friendly; raise on bigger GPUs
    gradient_accumulation_steps=2,        # effective batch 16
    per_device_eval_batch_size=8,
    num_train_epochs=150,
    fp16=USE_CUDA,                        # mixed precision on GPU
    dataloader_pin_memory=USE_CUDA,
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_steps=500,
    weight_decay=0.05,
    eval_strategy="steps", eval_steps=1000,
    save_strategy="steps", save_steps=1000,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="wer",          # best checkpoint on validation WER
    greater_is_better=False,
    save_total_limit=2,
    report_to="none",
)
_ok = set(inspect.signature(TrainingArguments.__init__).parameters)
_dropped = sorted(set(_want) - _ok)
if _dropped:
    print('dropped (not in this transformers version):', ', '.join(_dropped))
args = TrainingArguments(**{k: v for k, v in _want.items() if k in _ok})
print("model + args ready | trainable params:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

## 7 · Train, then save model + processor

In [ ]:
import inspect

# `tokenizer=` was renamed to `processing_class=` in transformers >= 4.46 — pick
# whichever the installed version accepts.
trainer_kwargs = dict(model=model, args=args, data_collator=data_collator,
                      train_dataset=train_ds, eval_dataset=valid_ds,
                      compute_metrics=compute_metrics)
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = processor.feature_extractor
else:
    trainer_kwargs["tokenizer"] = processor.feature_extractor
trainer = Trainer(**trainer_kwargs)

trainer.train()

out = os.path.join(MODELS, "kolsch_wav2vec2_model")
trainer.save_model(out); processor.save_pretrained(out)
print("saved ->", out, "| evaluate the test split in Notebook 6")

## Result

Selecting the best checkpoint on **validation WER** (not loss — CTC val loss
rebounds) gives ~15.2% val WER / 11.8% val CER, generalising to
**14.63% WER / 11.75% CER** on the held-out test set (Notebook 6).

**Quick GPU smoke test:** set `num_train_epochs=1` and `eval_steps=50` to confirm
the whole pipeline end-to-end before committing to the full run.